<a href="https://colab.research.google.com/github/mirandaurban/tc3009c/blob/main/A1_A01752391_Urban_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 · PyTorch: arquitectura y entrenamiento

**Modalidad:** individual

## Propósito

Comprobar que puedes distinguir entre:

- la **arquitectura del modelo**;
- el **mecanismo de entrenamiento**.

Trabajarás con FashionMNIST y usarás **el mismo training loop** para dos
arquitecturas distintas, y después para una tercera situación: la misma red con una
capa apagada. El loop no cambia ni una línea en todo el camino. Esa es la
idea.

Al final hay un modelo que **no corre**, y tu trabajo es averiguar por qué.

No se evalúa obtener la mejor accuracy ni hacer tuning.

> ### Cómo se contestan las preguntas de esta tarea
>
> Varias celdas de este notebook **imprimen shapes y totales**. Están ahí a
> propósito: son el ejemplo resuelto con el que compruebas tu método. Pero
> ninguna pregunta te pide ese número — todas te piden **la aritmética escrita**
> o **una arquitectura que este notebook no construye**. Copiar un total del
> output no contesta nada, y se nota de inmediato al calificar.

## Antes de empezar · cómo preparar el entorno

Tienes dos opciones. **Si no quieres instalar nada, usa Colab.**

### Opción A · Google Colab — la más rápida

1. Entra a [colab.research.google.com](https://colab.research.google.com).
2. **Archivo → Subir cuaderno** y elige este archivo `.ipynb`.
3. Listo. `torch` y `torchvision` ya vienen instalados; no tienes que hacer nada más.

### Opción B · Tu propia computadora

Necesitas **Python 3.9 o más nuevo**. Abre una terminal
(en Windows: *Símbolo del sistema* o *PowerShell*) y ejecuta:

```
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
```

Son unos **250 MB** y tarda varios minutos. Si te dice que `pip` no existe, prueba
con `python -m pip install ...` o, en Windows, `py -m pip install ...`.

**¿Por qué ese `--index-url`?** Instala la versión de PyTorch para **CPU**. Sin esa
opción, `pip` descarga la versión con soporte de GPU, que pesa cerca de **2.5 GB**
y aquí no hace falta: estas redes son pequeñas y entrenan en segundos sin tarjeta
gráfica.

Si usas **Anaconda**, ese mismo comando funciona dentro de tu entorno de conda.

> ⚠️ La celda del dataset **descarga FashionMNIST (~30 MB)** la primera vez.
> Después queda en la carpeta `data/` y ya no se vuelve a descargar. Empieza por
> ahí y no lo dejes para la última noche: si falla la descarga, no puedes avanzar.

**Cuánto tarda.** Vas a entrenar dos modelos, tres épocas cada uno. En una laptop
sin GPU eso es alrededor de **un minuto de cómputo en total**. Si tu máquina se
tarda muchísimo más, algo está mal: revísalo antes de seguir.

In [ ]:
# Comprobación del entorno. Si esta celda corre sin error, ya está todo listo.
import sys

print("Python      ", sys.version.split()[0])

try:
    import torch
    import torchvision
    print("torch       ", torch.__version__)
    print("torchvision ", torchvision.__version__)
    print("GPU         ", "sí" if torch.cuda.is_available() else "no (usaremos CPU, es suficiente)")
    print()
    print("Todo listo.")
except ImportError as e:
    print()
    print("FALTA UNA LIBRERÍA:", e.name)
    print("Revisa las instrucciones de instalación de la celda anterior,")
    print("o abre este notebook en Google Colab.")

Python       3.13.15
torch        2.11.0+cpu
torchvision  0.26.0+cpu
GPU          no (usaremos CPU, es suficiente)

Todo listo.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

torch.manual_seed(42)

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(
    training_data,
    batch_size=64,
    shuffle=True
)

loss_fn = nn.CrossEntropyLoss()

# Parte 1 · Modelo A: clasificación lineal

Primero usarás un modelo sin capas ocultas:

`784 → 10`

In [ ]:
model_a = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 10) # 28 * 28 = 784
)

optimizer_a = torch.optim.SGD(model_a.parameters(), lr=0.1)

print(model_a)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)


## Pregunta 1 · Shapes y parámetros

**(a)** Para el Modelo A, `784 → 10`, deriva el shape de `weight` y de `bias`, y
escribe el conteo de parámetros **como una suma, no como un total**. Así:

`(número × número) + número = ...`

**(b)** Explica por qué `nn.Linear` guarda `weight` como `[salidas, entradas]` y
no al revés.

**(c)** Ahora una red que **este notebook no construye**: la misma capa lineal,
pero para imágenes a color de `32 × 32 × 3` en lugar de `28 × 28` en escala de
grises, y con 100 clases en lugar de 10. Escribe la aritmética y el total.

> La celda de abajo verifica solo el inciso (a). El (b) y el (c) no salen de
> ningún output.

**Respuestas**

**(a)** Para el Modelo A, el weight es de 7840, el bias es de 10, por lo que la suma ponderada da como resultado 7850.

`(784 × 10) + 10 = 7850`

**(b)** `nn.Linear` guarda `weight` como [salidas, entradas] porque internamente, en la función matemática que usa, cada salida tiene un conjunto de pesos para todas las entradas, y esos pesos se utilizan en la multiplicación como $(xW^T)$.

**(c)** $(32 × 32 × 3 × 100 + 100= 307300)$ parámetros.




In [ ]:
# Verificación programática del Modelo A
for name, parameter in model_a.named_parameters():
    print(name, tuple(parameter.shape), parameter.numel())

print(
    "Total:",
    sum(p.numel() for p in model_a.parameters() if p.requires_grad)
)

1.weight (10, 784) 7840
1.bias (10,) 10
Total: 7850


# Parte 2 · Modelo B: MLP con dos capas ocultas

Construye esta arquitectura:

`784 → 256 → 64 → 10`

Usa ReLU después de cada capa oculta.

Completa únicamente los `TODO`.

In [ ]:
class ModelB(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        # TODO 1: 784 → 256
        self.hidden1 = nn.Linear(784, 256)

        # TODO 2: 256 → 64
        self.hidden2 = nn.Linear(256, 64)

        # TODO 3: 64 → 10
        self.output = nn.Linear(64, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)

        # TODO 4: primera capa oculta + ReLU
        h1 = self.hidden1(x)
        h1 = self.relu(h1)

        # TODO 5: segunda capa oculta + ReLU
        h2 = self.hidden2(h1)
        h2 = self.relu(h2)

        # TODO 6: logits de salida
        logits = self.output(h2)

        return logits


torch.manual_seed(42)
model_b = ModelB()

# --- dos comprobaciones, para que los errores digan algo útil ---------------

if not list(model_b.parameters()):
    raise RuntimeError(
        "ModelB no tiene ni un parámetro entrenable: faltan los TODO 1-3.\n"
        "Sustituye cada `...` por la capa nn.Linear que corresponda."
    )

_prueba = model_b(torch.zeros(2, 1, 28, 28))
if not torch.is_tensor(_prueba) or tuple(_prueba.shape) != (2, 10):
    raise RuntimeError(
        f"El forward devolvió {type(_prueba).__name__} {getattr(_prueba, 'shape', '')} "
        "y debería devolver un tensor [2, 10]: faltan los TODO 4-6."
    )

optimizer_b = torch.optim.SGD(model_b.parameters(), lr=0.1)

print(model_b)

ModelB(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden1): Linear(in_features=784, out_features=256, bias=True)
  (hidden2): Linear(in_features=256, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
)


## Pregunta 2 · Derivación del Modelo B

**(a)** Escribe el conteo de parámetros del Modelo B `784 → 256 → 64 → 10` **como
una suma capa por capa**, no como un total. Seis términos: el `weight` y el `bias`
de cada una de las tres capas.

**(b)** Ahora un MLP que **este notebook no construye**: `784 → 512 → 128 → 10`.
Deriva la suma completa y el total, sin escribir código.

**(c)** De los seis términos que escribiste en (a), **uno solo aporta más del 90%
del total**. Identifícalo y di cuánto aporta.

Ahora fíjate en esto: `hidden1` tiene 256 neuronas y `hidden2` tiene 64, o sea que
`hidden1` tiene **4 veces** más neuronas. Pero `hidden1` tiene **12 veces** más
parámetros que `hidden2`. ¿Por qué la diferencia en parámetros no es proporcional a
la diferencia en neuronas?

> La celda de abajo verifica solo el inciso (a).

**Respuesta:**

**(a)** El conteo de los parámetros se ve como
- $W_1 = (256×784) = 200704$
- $W_1 + B_1 = 200704 + 256 = 200 960$
- $W_2 = (64×256) = 16384$
- $W_2 + B_2 = 16384 + 64 = 16448$
- $W_O = (10×64) = 640$
- $W_O + B_0= = 640 + 10 = 650$
- $TOTAL = (W_1 + B_1) + (W_2 + B_2) + (W_O + B_0) = 218058$

**(b)**
- $W_1 = (512×784) = 401408$
- $W_1 + B_1 = 401408 + 512 = 401920$
- $W_2 = (128×512) = 65536$
- $W_2 + B_2 = 65536 + 128 = 65664$
- $W_O = (10×128) = 1280$
- $W_O + B_0= = 1280 + 10 = 1290$
- $TOTAL = (W_1 + B_1) + (W_2 + B_2) + (W_O + B_0) = 468874$

**(c)** El dominante es `hidden1.weight` con una aportación cercana al 92%.

La diferencia no es proporcional porque los parámetros de una capa nn.Linear son entradas × salidas (un producto), no solo la cantidad de neuronas.

$(200704/218058 ≈ 0.92)$


In [ ]:
# Verificación programática del Modelo B
for name, parameter in model_b.named_parameters():
    print(name, tuple(parameter.shape), parameter.numel())

print(
    "Total:",
    sum(p.numel() for p in model_b.parameters() if p.requires_grad)
)

hidden1.weight (256, 784) 200704
hidden1.bias (256,) 256
hidden2.weight (64, 256) 16384
hidden2.bias (64,) 64
output.weight (10, 64) 640
output.bias (10,) 10
Total: 218058


## Pregunta 3 · Cambio de arquitectura

Supón que `hidden2` cambia de **64 a 32 neuronas**, manteniendo lo demás igual.

Indica:

- qué parámetros cambiarían de shape;
- cuáles conservarían su shape;
- cuáles serían los nuevos shapes de los parámetros afectados.

Justifica tu respuesta a partir de cómo se conectan las capas.

**Respuesta:**
- Parámetros a cambiar: $W_2$, $B_2$ y $W_O$
- El resto de los parámetros continuaría con el mismo shape.
- Nuevos parámetros: $W_2 = (32, 256)$, $B_2=32$ y $W_O=(10, 32)$

La razón de estos cambios están dados en la forma en que se concetan las capas. Dado que se cambia la capa 2, se ven afectados los parámetros de esa capa y sus salidas. Por ende, los resultados alimentados por estas, los resultados de la capa siguiente también resultan afectados.


# Parte 3 · Gradientes en un batch

Usa un solo batch del Modelo B para generar gradientes.

In [ ]:
X, y = next(iter(train_dataloader))

optimizer_b.zero_grad()

# TODO 7: forward
pred = model_b(X)

# TODO 8: loss
loss = loss_fn(pred, y)

# TODO 9: backward
loss.backward()

print("Loss:", loss.item())
print()

for name, parameter in model_b.named_parameters():
    if parameter.grad is None:
        forma_grad = "sin gradiente — ¿ejecutaste backward()?"
    else:
        forma_grad = tuple(parameter.grad.shape)
    print(f"{name:20s}", "param:", tuple(parameter.shape), "grad:", forma_grad)

Loss: 2.302351474761963

hidden1.weight       param: (256, 784) grad: (256, 784)
hidden1.bias         param: (256,) grad: (256,)
hidden2.weight       param: (64, 256) grad: (64, 256)
hidden2.bias         param: (64,) grad: (64,)
output.weight        param: (10, 64) grad: (10, 64)
output.bias          param: (10,) grad: (10,)


## Pregunta 4 · Parámetro y gradiente

**(a)** Explica por qué `.grad` tiene **exactamente el mismo shape** que el
parámetro al que pertenece. La respuesta no es "porque PyTorch lo hace así":
di qué guarda cada número del gradiente.

**(b)** El batch de arriba tiene 64 imágenes. Si en vez de 64 tuviera **256**,
¿cambiaría el shape de `hidden1.weight.grad`? ¿Y cambiarían sus **valores**?
Contesta las dos por separado y explica la diferencia entre una y otra.

**(c)** `hidden1.weight` tiene 200,704 números y su gradiente también. Después de
`optimizer.step()`, ¿qué pasó con esos 200,704 gradientes? ¿Siguen ahí para el
siguiente batch?

> El output de arriba te da los shapes. Ninguno de los tres incisos se contesta
> con ellos.

**Respuesta:**

## Pregunta 4 · Parámetro y gradiente

**(a)** `.grad` tiene el mismo shape porque se hace una derivada parcial por cada parámetro, donde cada número del gradiente dice cuánto cambia la pérdida.

**(b)** No, no cambia el shape de `hidden1.weight.grad` porque este depende solo de cuántos parámetros tiene la capa. Lo que cambia son sus valores, ya que el gradiente es un promedio sobre las muestras del batch, entonces más muestras se traduce en distintas contribuciones y por ende, distintos números.

**(c)** `optimizer.step()` lee los gradientes que `autograd` dejó en `.grad` y actualiza cada peso. Los gradientes no se borran al hacer step(), por eso el loop llama a `optimizer.zero_grad()` al inicio de cada iteración, sin esa llamada, los gradientes del siguiente batch se acumularían sobre los del anterior en vez de reemplazarlos.

# Parte 4 · Un mismo training loop

La función de entrenamiento ya está completa. No debes modificarla.

Comprueba que puede entrenar **ambos modelos** sin cambiar su lógica.

In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0.0

    for X, y in dataloader:
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# Modelo A
for epoch in range(3):
    avg_loss = train_one_epoch(
        train_dataloader,
        model_a,
        loss_fn,
        optimizer_a
    )
    print(f"Modelo A · Epoch {epoch + 1}: loss={avg_loss:.4f}")

Modelo A · Epoch 1: loss=0.6251
Modelo A · Epoch 2: loss=0.4887
Modelo A · Epoch 3: loss=0.4642


In [ ]:
# Modelo B
for epoch in range(3):
    avg_loss = train_one_epoch(
        train_dataloader,
        model_b,
        loss_fn,
        optimizer_b
    )
    print(f"Modelo B · Epoch {epoch + 1}: loss={avg_loss:.4f}")

Modelo B · Epoch 1: loss=0.6839
Modelo B · Epoch 2: loss=0.4359
Modelo B · Epoch 3: loss=0.3848


## Pregunta 5 · La idea central

El Modelo A es un clasificador lineal y el Modelo B es una MLP con dos capas ocultas.

¿Por qué `train_one_epoch()` puede entrenar ambos sin cambiar su lógica?

Relaciona en tu explicación:

- `model(X)`;
- parámetros;
- autograd;
- `optimizer.step()`.

**Respuesta:**
Porque la función `train_one_epoch()` funciona como un contenedor para las funciones clave, las cuales comparten ambos modelos por la forma en la que se construyeron. Por ejemplo:
- `model(X)` produce logits independientemente del número de capas que tenga.
- `model.parameters()` devuelve los valores obtenidos en el entrenamiento independientemente del modelo, ambos están parametrizados, entonces es apto para ambos.
- `loss.backward()` calcula los gradientes de los parámetros dado el grafo construido en forward. Funciona con cualquier modelo parametrizado, entonces funciona para ambos casos.
- `optimizer.step()` aplica la misma regla de actualización a cada parámetro que recibe, sin distinguir capas, por eso funciona con ambos modelos.


# Parte 5 · Apagar una capa

Hasta aquí **todos** los parámetros del Modelo B eran entrenables. No tiene por qué
ser así: en PyTorch, "ser entrenable" es un interruptor que vive en cada tensor y se
llama `requires_grad`.

Vas a apagar **`hidden2`**, la capa de en medio, y dejar el resto igual. Nada más
cambia: mismo modelo, mismo optimizador, mismo training loop.

> Esto no es un truco de laboratorio. Congelar capas es **exactamente** lo que se
> hace al reaprovechar un modelo ya entrenado, y volverá más adelante en el curso.

## Pregunta 6 · Predice antes de ejecutar

**Escribe tus predicciones antes de correr las celdas de abajo.** Esta pregunta
solo funciona en ese orden, y una predicción equivocada y razonada vale más que
una respuesta correcta escrita después de ver la tabla.

1. Con `hidden2` congelada, ¿cuántos parámetros entrenables quedan? **Escribe la
   resta**, no el total.
2. Después de `backward()`, ¿qué esperas encontrar en `hidden2.weight.grad`?
   ¿Y en `hidden1.weight.grad`, que está **más atrás** en la red?
3. Después de entrenar una época, ¿qué parámetros se habrán movido y cuáles no?
4. Y una que no vas a poder verificar corriendo nada: si en lugar de `hidden2`
   congeláramos **`output`**, ¿cuántos entrenables quedarían? Escribe la resta.

**Respuesta:**
1. En ese caso, la resta se vería de esta forma:
- Total antes: $218,058$
- hidden2.weight: $64 × 256 = 16,384$
- hidden2.bias:   $64, $
- Entrenables $= 218,058 − 16,384 − 64 = 201,610$

2. En`hidden2.weight.grad` se espera un valor de `None`, pues al estar congelada no acumula gradiente en sus parámetros, aunque la capa sí participa en el forward. En `hidden1.weight.grad` el backward atraviesa `hidden2` para llegar a `hidden1` por lo que su shape sería (256, 784).

3. En la capa `hidden1` se habrán movido los parámetros de weight y bias, al igual que en la capa `output`. Los de la capa `hidden2` permanecerán intáctos, pues al estar congelada, no se le realizan cambios con los gradientes.

4. Si congelamos `output`: $entrenables = 218,058 − (10×64) − 10 = 218,058 − 640 − 10 = 217,408$

In [ ]:
# TODO 10: congela hidden2. Pon requires_grad en False en sus DOS parámetros,
#          weight y bias. Puedes hacerlo con una línea por parámetro o con un
#          for sobre model_b.hidden2.parameters().

for param in model_b.hidden2.parameters():
    param.requires_grad = False


print("¿Entrenable?")
for nombre, parametro in model_b.named_parameters():
    print(f"   {nombre:20s} {parametro.requires_grad}")

entrenables = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
print()
print(f"Parámetros entrenables ahora: {entrenables:,}")
print("Antes de congelar eran 218,058. ¿Coincide con tu predicción?")

¿Entrenable?
   hidden1.weight       True
   hidden1.bias         True
   hidden2.weight       False
   hidden2.bias         False
   output.weight        True
   output.bias          True

Parámetros entrenables ahora: 201,610
Antes de congelar eran 218,058. ¿Coincide con tu predicción?


### Una época más, con la capa apagada

La celda siguiente guarda una copia de todos los parámetros, entrena una época con
**el mismo `train_one_epoch` de siempre**, y después mide cuánto se movió cada uno.

Fíjate en las tres columnas juntas: son la respuesta a tu predicción.

In [ ]:
antes = {n: p.detach().clone() for n, p in model_b.named_parameters()}

perdida = train_one_epoch(train_dataloader, model_b, loss_fn, optimizer_b)
print(f"Pérdida promedio de esta época: {perdida:.4f}")
print()

print(f"{'parámetro':20s} {'entrenable':>11s} {'tiene .grad':>12s} {'se movió':>12s}")
print("-" * 60)
for nombre, parametro in model_b.named_parameters():
    movido = (parametro.detach() - antes[nombre]).abs().max().item()
    print(f"{nombre:20s}"
          f" {str(parametro.requires_grad):>11s}"
          f" {('no' if parametro.grad is None else 'sí'):>12s}"
          f" {movido:>12.6f}")

Pérdida promedio de esta época: 0.3526

parámetro             entrenable  tiene .grad     se movió
------------------------------------------------------------
hidden1.weight              True           sí     0.032551
hidden1.bias                True           sí     0.040547
hidden2.weight             False           no     0.000000
hidden2.bias               False           no     0.000000
output.weight               True           sí     0.082986
output.bias                 True           sí     0.027718


## Pregunta 7 · El gradiente que pasa de largo

Mira otra vez la tabla. `hidden2` **no guarda gradiente propio** — y sin embargo
`hidden1`, que está **más atrás** en la red, sí recibió el suyo y se movió.

Contesta:

1. Si el backward va de la salida hacia la entrada, tuvo que **atravesar** `hidden2`
   para llegar a `hidden1`. ¿Qué usó para atravesarla, si esa capa ya no acumula
   gradiente?
2. ¿Qué dos cosas distintas está haciendo autograd al pasar por una capa, y cuál de
   las dos dejó de hacer al congelarla?
3. El optimizador sigue recibiendo `model_b.parameters()`, que **incluye** a
   `hidden2`. ¿Por qué entonces `step()` no la movió?

> Pista para 1 y 2: en S02 distinguimos `∂L/∂h` de `∂L/∂W`. Una era el camino y la
> otra el destino. Aquí se ve por qué había que distinguirlas.
>
> Pista para 3: es el mismo comportamiento que hace que la primera iteración del
> loop roto de la actividad de clase **no** truene.

**Respuesta:**
1. La capa no acumula gradiente en sus parámetros, pero sí se utiliza en el cálculo de la derivada que atraviesa. En el backward se utilizan los pesos como constantes.

2. Al pasar por una capa, calcula la derivada que se propaga en todo el proceso y acumula esa derivada del parámetro. Al congelar una capa, lee la función de `requires_grad=False` de los parámetros y no acumula la derivada de esos parámetros, pero sí calcula la derivada que se propaga hacia atrás para que las capas anteriores puedan recibir su gradiente.

3. `step()` lee los valores acumulados y actualiza conforme a los cálculos de la derivada. Al no haber sido guardados los valores de la capa `hidden2` no hay elementos con que actualizar, por lo que permanece igual.

# Parte 6 · Un modelo que no corre

Hasta aquí todo lo que construiste funcionó. En la práctica, la mitad del trabajo
con arquitecturas es leer un error de shapes y saber **qué capa lo causó**.

La celda de abajo define `ModelC` y **está mal a propósito**. No la arregles
todavía: córrela primero y lee el mensaje completo.

In [ ]:
class ModelC(nn.Module):
    """Este modelo NO corre. Es el ejercicio."""

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(784, 256)
        self.hidden2 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        h1 = self.relu(self.hidden1(x))
        h2 = self.relu(self.hidden2(h1))
        logits = self.output(h2)
        return logits


model_c = ModelC()

# Un batch pequeño, solo para provocar el error. No entrena nada.
entrada = torch.zeros(8, 1, 28, 28)

try:
    model_c(entrada)
    print("No falló. Si ves esto, alguien ya arregló el modelo.")
except RuntimeError as e:
    print("FALLÓ, como debía. Mensaje completo:")
    print()
    print(e)

FALLÓ, como debía. Mensaje completo:

mat1 and mat2 shapes cannot be multiplied (8x256 and 128x64)


## Pregunta 8 · Diagnóstico

Sin cambiar nada todavía, contesta leyendo **solo el mensaje de error**:

1. El mensaje menciona dos shapes que no se pueden multiplicar. **Copia los dos
   números que se contradicen** y di de dónde sale cada uno: uno lo produjo una
   capa, el otro lo espera otra.
2. ¿Cuáles son las **dos capas** involucradas? Nómbralas por su nombre en el
   código.
3. El error aparece en `hidden2`. ¿Significa eso que `hidden2` es la capa
   equivocada? Justifica: puede que sí, puede que no.

> Fíjate en el 8 del batch: aparece en el mensaje y **no** es parte del problema.
> Saber cuál número ignorar es la mitad de leer un error de shapes.

**Respuesta:**
1. Los números que se contradicen son 256 (salida de `hidden1`) y 128 (entrada esperada por `hidden2`). El 8 del principio es el batch size y no es parte del conflicto.

2. Son `hidden1` y `hidden2`

3. El error aparece en `hidden2` porque es la capa que recibe un tamaño incompatible, pero eso no implica que esta capa sea la equivocada. El problema puede estar en `hidden1` (si debería producir 128) o en `hidden2` (si debería esperar 256). El mensaje solo dice dónde falla y al no tener más información no es posible determinar cuál está mal definida, solo que no son compatibles.

## Pregunta 9 · Los dos arreglos

Este error tiene **dos arreglos distintos**, y los dos hacen que el modelo corra.

1. Escríbelos los dos: qué línea cambiarías en cada caso y a qué.
2. **No son equivalentes.** Uno conserva la arquitectura `784 → 256 → 64 → 10` y
   el otro produce una red distinta. Di cuál es cuál.
3. Calcula el total de parámetros que resulta de **cada** arreglo. Escribe las dos
   sumas. ¿Cuál de las dos redes tiene más capacidad?
4. Si nadie te dijera qué arquitectura se quería, ¿cómo decidirías cuál arreglo es
   el correcto? No hay una sola respuesta buena aquí; se evalúa el criterio.

**Respuesta:**

1. Las dos posibnles soluciones son:
```python
# Opción 1
def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(784, 256)
        self.hidden2 = nn.Linear(256, 64)
        self.output = nn.Linear(64, 10)
        self.relu = nn.ReLU()

# Opción 2
def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(784, 128)
        self.hidden2 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 10)
        self.relu = nn.ReLU()
```

2. La primera opción conserva la arquitectura `784 → 256 → 64 → 10` y la otra produce una red distinta, `784 → 128 → 64 → 10`
3. El total de parámetros se ve la siguiente manera:

- hidden1.weight: $(128 × 784) = 100352$
- hidden1.bias:   $128$
- hidden2.weight: $(64 × 128) = 8,192$
- hidden2.bias:   $64$
- output.weight:  $(10 × 64) = 640$
- output.bias:    $10$


$$
Total = (128×784) + 128 + (64×128) + 64 + (10×64) + 10
      = 100,352 + 128 + 8,192 + 64 + 640 + 10
      = 109,386
$$

La Opción 1 (218,058 vs 109,386). Tiene más parámetros, sobretodo en hidden1 por lo que podría representarla mejor, pero con el riesgo de overfitting.

4. Debería evaluar el problema completo: ¿si hay un documentación de referencia con el que comparar? ¿la elección de esa arquitectura no funcional fue un typo o por qué se escribió de esa manera? ¿para qué se está diseñando, su objetivo? ¿con qué presupuesto se cuenta en términos de cómputo? ¿cuál es el tamaño del dataset? Básicamente entender la arquitectura completa y con base en ello, elegir cuál es la opción óptima.

In [ ]:
# TODO 11: arregla ModelC. Escribe la versión que CONSERVA la arquitectura
#          784 -> 256 -> 64 -> 10, es decir la que elegiste en la Pregunta 9.
#          Copia las cuatro capas de ModelC y cambia solo la que está mal.


class ModelCArreglado(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(784, 256)
        self.hidden2 = nn.Linear(256, 64)
        self.output = nn.Linear(64, 10)
        self.relu = nn.ReLU()

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        h1 = self.relu(self.hidden1(x))
        h2 = self.relu(self.hidden2(h1))
        logits = self.output(h2)
        return logits


model_c = ModelCArreglado()

if not list(model_c.parameters()):
    raise RuntimeError(
        "ModelCArreglado no tiene ni un parámetro: falta el TODO 11a.\n"
        "Escribe las cuatro capas, con la corrección que decidiste en la P9."
    )

salida = model_c(torch.zeros(8, 1, 28, 28))

if not torch.is_tensor(salida):
    raise RuntimeError(
        "El forward no devolvió un tensor: falta el `return` del TODO 11b."
    )

total = sum(p.numel() for p in model_c.parameters() if p.requires_grad)

print("Shape de salida:", tuple(salida.shape), " (debe ser (8, 10))")
print(f"Parámetros entrenables: {total:,}")
print()
if tuple(salida.shape) == (8, 10):
    print("Corre. Compara el total con la suma que escribiste en la Pregunta 9.")
else:
    print("La salida no es (8, 10). Revisa la última capa.")

Shape de salida: (8, 10)  (debe ser (8, 10))
Parámetros entrenables: 218,058

Corre. Compara el total con la suma que escribiste en la Pregunta 9.


# Entregable

Entrega **este mismo notebook**:

- completa todos los `TODO`;
- responde las nueve preguntas;
- conserva los outputs visibles;
- verifica que corre de principio a fin sin errores.

Nómbralo:

`A1_<Matricula>_<Apellido>.ipynb`

No se asignan puntos por obtener una accuracy o una loss específica.

## Criterios de evaluación

| Evidencia | Puntos |
|---|---:|
| Construcción correcta del Modelo B (TODO 1–6) | 10 |
| Derivación de shapes y parámetros (P1, P2) | 15 |
| Efecto de cambiar una capa (P3) | 10 |
| Parámetro y gradiente (P4) | 10 |
| Por qué el mismo loop entrena ambos (P5) | 10 |
| Congelar una capa: predicción y lectura (P6) | 10 |
| El gradiente que pasa de largo (P7) | 10 |
| Diagnóstico del error de shapes (P8) | 10 |
| Los dos arreglos y su criterio (P9, TODO 11) | 10 |
| Notebook ejecutable y claro | 5 |
| **Total** | **100** |

## Uso de herramientas de IA

Puedes apoyarte en asistentes de IA para dudas de sintaxis. Lo que esta tarea evalúa
son **tus derivaciones y tus explicaciones**: si no puedes sostener en voz alta lo que
escribiste en las nueve respuestas, la tarea no cumplió su propósito.

Declara abajo qué herramientas usaste y para qué. Declararlo no baja la
calificación; no declararlo sí es un problema.

- Herramienta: DeepSeek
- Uso realizado: Asistencia en la modificación y adaptación del código proporcionado por mí. Concretamente se le proporcionó pseudocódigo o instrucciones específicas sobre los cambios requeridos y su propósito, y se utilizó la herramienta para proponer e implementar dichas modificaciones.

- Secciones donde se utilizó: Todas las secciones de la actividad en las que se requirieron modificaciones, implementación o ajustes de código.

- Validación realizada por el estudiante: Se revisó el código generado o modificado, verificando que correspondiera con las instrucciones y requerimientos de la actividad, así como su integración con el código proporcionado previamente.

- Confirmación: Confirmo que revisé, verifiqué y asumo la responsabilidad del contenido final entregado.

## Antes de entregar

Reinicia el entorno y ejecuta todas las celdas desde cero.

Confirma que no hay errores, que no quedan `TODO` sin resolver y que guardaste el
notebook con los outputs visibles.

Las preguntas **6 y 7** se contestan con lo que tú predijiste y observaste. Si
contestaste la 6 después de ver el resultado, la tarea no hizo su trabajo — y se nota
al leerla.

La **8** y la **9** se contestan leyendo un error, no corriendo código de más.
Si arreglaste `ModelC` antes de contestarlas, perdiste la pregunta.

In [ ]:
# Autocomprobación del código. No evalúa tus respuestas escritas.
problemas = []

if not isinstance(globals().get("model_b"), nn.Module):
    problemas.append("Falta construir model_b.")

if "train_one_epoch" not in globals():
    problemas.append("Falta train_one_epoch.")

try:
    prueba = torch.zeros(2, 1, 28, 28)
    for nombre, m in (("Modelo A", model_a), ("Modelo B", model_b)):
        salida = m(prueba)
        if not torch.is_tensor(salida) or tuple(salida.shape) != (2, 10):
            problemas.append(f"{nombre} no produce [2, 10]. Revisa su forward.")
except Exception as e:
    problemas.append(f"Los modelos no corren: {type(e).__name__}: {e}")

# Parte 5: ¿se congeló hidden2?
try:
    if model_b.hidden2.weight.requires_grad:
        problemas.append("Parte 5: hidden2 sigue entrenable. ¿Completaste el TODO 10?")
except Exception:
    pass

# Parte 6: ¿arreglaste ModelC?
try:
    _c = globals().get("model_c")
    if _c is None:
        problemas.append("Parte 6: falta model_c. ¿Completaste el TODO 11?")
    else:
        _s = _c(torch.zeros(2, 1, 28, 28))
        if tuple(_s.shape) != (2, 10):
            problemas.append("Parte 6: model_c no produce [2, 10].")
except RuntimeError:
    problemas.append(
        "Parte 6: model_c sigue tronando. ¿Completaste el TODO 11?")
except Exception:
    pass

# Parte 3: ¿quedaron gradientes?
try:
    # hidden2 queda sin gradiente a propósito desde la Parte 5; el resto no.
    sin_grad = [n for n, p in model_b.named_parameters()
                if p.grad is None and not n.startswith("hidden2")]
    if sin_grad:
        problemas.append(
            "Parte 3: no hay gradientes en " + ", ".join(sin_grad[:3])
            + ". ¿Completaste los TODO 7-9?"
        )
except Exception:
    pass

print("=" * 62)
if problemas:
    print("REVISA:")
    for p in problemas:
        print("  -", p)
else:
    print("La estructura del código está completa.")
print("=" * 62)
print()
print("Esto NO revisa tus nueve respuestas escritas, que es lo que más pesa.")
print("Reléelas: cada una debe derivar o explicar algo, no repetir un output.")

La estructura del código está completa.

Esto NO revisa tus nueve respuestas escritas, que es lo que más pesa.
Reléelas: cada una debe derivar o explicar algo, no repetir un output.


---

Basado en ejemplos y ejercicios de **Dive into Deep Learning** y en el **Quickstart oficial de PyTorch**.  
*Deep Learning · TC3009C · Tecnológico de Monterrey*